In [1]:
# Cell 1: Setup and Imports
import os
import sys
import yaml
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# Safely resolve project root
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.models.cb_model import ContentBasedRecommender
from src.data_pipeline.preprocess import clean_metadata_text

print("[INFO] Imports loaded successfully.")

[INFO] Imports loaded successfully.


In [2]:
# Cell 2: Load Configuration and Metadata
CONFIG_PATH = os.path.join(PROJECT_ROOT, 'configs', 'data_config.yaml')

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    data_config = yaml.safe_load(f)

PROCESSED_DIR = os.path.join(PROJECT_ROOT, data_config['paths']['processed_data'])
META_FILE = data_config['paths']['metadata_file']
META_PATH = os.path.join(PROCESSED_DIR, META_FILE)

print(f"[INFO] Loading metadata from: {META_PATH}")
meta_df = pd.read_parquet(META_PATH)

print(f"[INFO] Metadata loaded. Shape: {meta_df.shape}")

[INFO] Loading metadata from: d:\Ahmed\study\DEPI\tasks\Final_project\recommendation-system\data/processed/metadata_processed.parquet
[INFO] Metadata loaded. Shape: (10000, 10)


In [3]:
# Cell 3: Clean and Combine Text Features
print("[INFO] Applying text cleaning...")
text_cols = ['title', 'categories', 'features', 'description']

for col in text_cols:
    meta_df[col] = meta_df[col].apply(clean_metadata_text)

print("[INFO] Combining features into 'combined_text'...")
meta_df['combined_text'] = (
    meta_df['title'] + " " +
    meta_df['categories'] + " " +
    meta_df['features'] + " " +
    meta_df['description']
).str.lower()

# Display a sample of the cleaned text
pd.set_option('display.max_colwidth', 150)
display(meta_df[['parent_asin', 'combined_text']].head(2))

[INFO] Applying text cleaning...
[INFO] Combining features into 'combined_text'...


,parent_asin,combined_text
0,B00JX3Q28Y,plugable usb 3.0 sharing switch for one-button swapping of usb device or hub between two computers (ab switch) 'electronics' 'computers & accessor...
1,B06XKRXLDR,khomo - ipad 2 3 and 4 generation case - dual series - super slim black cover with rubberized back and smart auto wake sleep feature for apple ipa...


In [4]:
# Cell 4: Initialize and Fit Content-Based Recommender
print("[INFO] Initializing ContentBasedRecommender...")
cb_model = ContentBasedRecommender(
    max_features=15000, 
    ngram_range=(1, 2), 
    stop_words='english'
)

print("[INFO] Fitting model (TF-IDF Vectorization)...")
cb_model.fit_items(meta_df, text_column='combined_text', item_column='parent_asin')

print(f"[INFO] TF-IDF Matrix Shape: {cb_model.tfidf_matrix.shape}")

[INFO] Initializing ContentBasedRecommender...
[INFO] Fitting model (TF-IDF Vectorization)...
[INFO] Content vectors loaded onto device: cuda
[INFO] TF-IDF Matrix Shape: (10000, 15000)


In [5]:
# Cell 5: Generate Test Recommendations
# We will test on index 1 (KHOMO iPad Case) based on previous EDA
test_asin = meta_df['parent_asin'].iloc[1]
target_title = meta_df[meta_df['parent_asin'] == test_asin]['title'].values[0]

print(f"Target Product ASIN: {test_asin}")
print(f"Target Title: {target_title[:100]}...")
print("-" * 80)

# Get top 5 recommendations
recommendations = cb_model.get_similar_items(target_asin=test_asin, top_k=5)

for i, rec in enumerate(recommendations, 1):
    rec_title = meta_df[meta_df['parent_asin'] == rec['parent_asin']]['title'].values[0]
    print(f"{i}. Match Score: {rec['similarity_score']:.4f} | ASIN: {rec['parent_asin']}")
    print(f"   Title: {rec_title[:100]}...")

Target Product ASIN: B06XKRXLDR
Target Title: KHOMO - iPad 2 3 and 4 Generation Case - DUAL Series - Super Slim Black Cover with Rubberized back a...
--------------------------------------------------------------------------------
1. Match Score: 0.8985 | ASIN: B06X3SGR9Q
   Title: KHOMO iPad Air 2 Case - Dual Series - Ultra Slim Cover with Auto Sleep Wake Feature for Apple iPad A...
2. Match Score: 0.8732 | ASIN: B07WNJQFP9
   Title: iPad Mini 1 2 & 3 Case - DUAL Blue Cover with Rubberized back and Smart Feature for Apple iPad Mini ...
3. Match Score: 0.4893 | ASIN: B074V9XVZC
   Title: KHOMO - iPad Pro 10.5 Inch & iPad Air 3 2019 Pink Color Case - Companion Cover - Perfect match for A...
4. Match Score: 0.4407 | ASIN: B087TWDMCC
   Title: KHOMO - iPad Pro 10.5 Inch & iPad Air 3 2019 Hybrid Clear Case With Pen Holder - Companion Cover - P...
5. Match Score: 0.2839 | ASIN: B07DWQTSQM
   Title: ProCase iPad 2 3 4 Case (Old Model) – Ultra Slim Lightweight Stand Case with Translucent Fros

In [6]:
# Cell 6: Evaluate Content-Based Model on Validation Set
from mlops.train import prepare_sparse_matrices
from mlops.evaluate import evaluate_model_at_k

# 1. Load interaction datasets
TRAIN_PATH = os.path.join(PROCESSED_DIR, data_config['paths']['train_file'])
VAL_PATH = os.path.join(PROCESSED_DIR, data_config['paths']['val_file'])

print("[INFO] Loading interaction datasets...")
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)

# Filter for 2017+ to match CF comparison logic
train_df = train_df[train_df['timestamp'] >= '2017-01-01']

# 2. Extract exact item order used in sparse matrices
print("[INFO] Aligning Metadata with Interaction Data using original ASINs...")
all_interactions = pd.concat([train_df, val_df])

# Get the exact order of encoded items as they will appear in the sparse matrix
cf_items_encoded = all_interactions['parent_asin'].unique()

# Build a mapping dictionary: Encoded Integer -> Original String ASIN
mapping_df = all_interactions.drop_duplicates('parent_asin')
encoded_to_original = dict(zip(mapping_df['parent_asin'], mapping_df['parent_asin_original']))

# Create an ordered list of original string ASINs matching the sparse matrix order
cf_items_original = [encoded_to_original[encoded] for encoded in cf_items_encoded]

# Reorder metadata to match the CF matrix item indices perfectly
meta_df_aligned = meta_df.set_index('parent_asin').loc[cf_items_original].reset_index()

# Refit the item TF-IDF matrix with the perfectly aligned metadata
cb_model.fit_items(meta_df_aligned, text_column='combined_text', item_column='parent_asin')

# 3. Prepare sparse matrices
print("[INFO] Preparing sparse matrices...")
train_matrix, val_matrix, eval_users = prepare_sparse_matrices(train_df, val_df)

# 4. Fit User Profiles in CB Model
print("[INFO] Fitting user profiles...")
cb_model.fit(train_matrix)

# 5. Evaluate using the standard pipeline
print("\n[INFO] Evaluating Content-Based Model at K=10...")
cb_metrics = evaluate_model_at_k(cb_model, val_matrix, eval_users, k=10)

print("\n" + "="*40)
print("Content-Based Model Results (Val Set)")
print("="*40)
for metric, value in cb_metrics.items():
    print(f"{metric}: {value:.4f}")

[INFO] Loading interaction datasets...
[INFO] Aligning Metadata with Interaction Data using original ASINs...
[INFO] Content vectors loaded onto device: cuda
[INFO] Preparing sparse matrices...
[INFO] Fitting user profiles...

[INFO] Evaluating Content-Based Model at K=10...



Content-Based Model Results (Val Set)
HitRate_10: 0.0176
Precision_10: 0.0018
Adj_Precision_10: 0.0124
Recall_10: 0.0124
MRR_10: 0.0056
NDCG_10: 0.0064
